In [ ]:
# Basic
!pip install pandas numpy matplotlib scikit-learn torch
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# Path
train_path = "train_FD001.txt"

# Load train data (handle extra spaces)
df_train = pd.read_csv(train_path, sep="\s+", header=None)
df_train.dropna(axis=1, how='all', inplace=True)

# Assign column names dynamically
n_cols = df_train.shape[1]
df_train.columns = (
    ['engine_unit', 'cycle', 'setting1', 'setting2', 'setting3'] +
    [f'sensor{i}' for i in range(1, n_cols - 5 + 1)]  # 2 IDs + 3 settings
)

print(f"Train shape: {df_train.shape}")
print(df_train.head())



In [ ]:
# Max cycle per engine
max_cycles = df_train.groupby('engine_unit')['cycle'].max().reset_index()
max_cycles.columns = ['engine_unit', 'max_cycle']

# Merge to compute RUL
df_train = pd.merge(df_train, max_cycles, on='engine_unit')
df_train['RUL'] = df_train['max_cycle'] - df_train['cycle']
df_train.drop('max_cycle', axis=1, inplace=True)

print(df_train[['engine_unit','cycle','RUL']].head())


In [ ]:
scaler = MinMaxScaler(feature_range=(-1,1))
df_train_scaled = df_train.copy()

cols_to_scale = df_train.columns.difference(['engine_unit', 'cycle', 'RUL'])
df_train_scaled[cols_to_scale] = scaler.fit_transform(df_train[cols_to_scale])

print("Scaled training data ready")


In [ ]:
class CMAPSSLSTMDataset(Dataset):
    def __init__(self, df, seq_len=30):
        self.seq_len = seq_len
        self.data = []

        # Group by engine
        for eid, group in df.groupby('engine_unit'):
            group = group.sort_values('cycle')
            values = group.drop(['engine_unit', 'cycle', 'RUL'], axis=1).values
            rul = group['RUL'].values

            for i in range(len(group) - seq_len + 1):
                self.data.append((values[i:i+seq_len], rul[i+seq_len-1]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        X, y = self.data[idx]
        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [ ]:
seq_len = 30
train_dataset = CMAPSSLSTMDataset(df_train_scaled, seq_len=seq_len)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"Total training sequences: {len(train_dataset)}")


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)      # (batch, seq, hidden)
        out = out[:, -1, :]        # take last timestep
        out = self.fc(out)         # (batch, 1)
        return out.squeeze()


In [ ]:
input_size = df_train_scaled.drop(['engine_unit','cycle','RUL'], axis=1).shape[1]
model = LSTMModel(input_size).to(device)  # Move model to device

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 30
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)  # Move batch to device
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}")


In [ ]:
# Load test data
test_path = "test_FD001.txt"
df_test = pd.read_csv(test_path, sep="\s+", header=None)
df_test.dropna(axis=1, how='all', inplace=True)
df_test.columns = df_train.columns[:-1]  # same as train (no RUL)

# Load true RULs (last cycle)
rul_truth = pd.read_csv("RUL_FD001.txt", sep="\s+", header=None)
rul_truth.columns = ["RUL"]

# Scale sensors/settings
df_test_scaled = df_test.copy()
cols_to_scale = df_test_scaled.columns.difference(['engine_unit','cycle'])
df_test_scaled[cols_to_scale] = scaler.transform(df_test_scaled[cols_to_scale])

# Select last seq_len cycles per engine
last_sequences = []
y_true = []

for i, (eid, group) in enumerate(df_test_scaled.groupby('engine_unit')):
    group = group.sort_values('cycle')
    if len(group) >= seq_len:
        seq = group.drop(['engine_unit','cycle'], axis=1).values[-seq_len:]
        last_sequences.append(seq)
        y_true.append(rul_truth.iloc[i,0])

# Prepare test data
X_test = torch.tensor(np.array(last_sequences), dtype=torch.float32).to(device)  # Move to device
y_true = np.array(y_true, dtype='float32')

print(f"Prepared test sequences: {len(X_test)}")


In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test).cpu().numpy()  # Move output to CPU for numpy

# Success accuracy: % of predictions within ±5 of true RUL
success_mask = np.abs(y_true - y_pred) <= 10
success_accuracy = 100 * np.mean(success_mask)

print(f"Success Accuracy (±5): {success_accuracy:.2f}%")

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

accuracy_per_engine = 100 * (1 - np.abs(y_true - y_pred) / y_true)
accuracy_per_engine = np.clip(accuracy_per_engine, 0, 100)
mean_accuracy = accuracy_per_engine.mean()

print(f"Test MAE: {mae:.2f}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Relative Accuracy: {mean_accuracy:.2f}%")